<a href="https://colab.research.google.com/github/jAgaThasweety/rabacademy/blob/main/retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# KPI DICTIONARY & DATA QUALITY CONTRACT
# Retail Orders Project
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

FILE = "retail-orders-raw.csv"

df = pd.read_csv(FILE)

print("=" * 70)
print("RETAIL KPI & DATA QUALITY PROJECT")
print("=" * 70)

print(f"\nRows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\nColumn Names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nStandardized Columns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. AUTOMATICALLY IDENTIFY COMMON COLUMNS
# ------------------------------------------------------------

def find_column(possible_names):

    for name in possible_names:
        if name in df.columns:
            return name

    return None


order_col = find_column([
    "transaction_id",
    "order_id",
    "orderid",
    "transactionid"
])

customer_col = find_column([
    "customer_id",
    "customerid",
    "customer"
])

quantity_col = find_column([
    "quantity",
    "qty",
    "units"
])

price_col = find_column([
    "price_per_unit",
    "unit_price",
    "price",
    "selling_price"
])

total_col = find_column([
    "total_spent",
    "total_amount",
    "total",
    "sales",
    "revenue",
    "amount"
])

date_col = find_column([
    "transaction_date",
    "order_date",
    "date",
    "orderdate"
])

category_col = find_column([
    "category",
    "product_category"
])

discount_col = find_column([
    "discount",
    "discount_amount",
    "discount_percentage"
])


print("\nDetected Columns:")
print("Order     :", order_col)
print("Customer  :", customer_col)
print("Quantity  :", quantity_col)
print("Price     :", price_col)
print("Total     :", total_col)
print("Date      :", date_col)
print("Category  :", category_col)
print("Discount  :", discount_col)


# ------------------------------------------------------------
# 4. DATA QUALITY - COMPLETENESS
# ------------------------------------------------------------

missing_count = df.isnull().sum()
missing_percent = df.isnull().mean() * 100

completeness = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": missing_count.values,
    "Missing %": missing_percent.values
})

print("\n" + "=" * 70)
print("COMPLETENESS CHECK")
print("=" * 70)

display(completeness)


# ------------------------------------------------------------
# 5. DATA QUALITY - UNIQUENESS
# ------------------------------------------------------------

if order_col:

    duplicate_count = df[order_col].duplicated().sum()
    duplicate_rate = duplicate_count / len(df) * 100

else:

    duplicate_count = np.nan
    duplicate_rate = np.nan


# ------------------------------------------------------------
# 6. DATA QUALITY - VALIDITY
# ------------------------------------------------------------

invalid_quantity = 0
invalid_price = 0
invalid_total = 0
invalid_dates = 0

if quantity_col:

    numeric_quantity = pd.to_numeric(
        df[quantity_col],
        errors="coerce"
    )

    invalid_quantity = (numeric_quantity <= 0).sum()


if price_col:

    numeric_price = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    invalid_price = (numeric_price <= 0).sum()


if total_col:

    numeric_total = pd.to_numeric(
        df[total_col],
        errors="coerce"
    )

    invalid_total = (numeric_total < 0).sum()


if date_col:

    converted_dates = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    invalid_dates = converted_dates.isna().sum()


# ------------------------------------------------------------
# 7. DATA QUALITY - CONSISTENCY
# ------------------------------------------------------------

consistency_failures = np.nan

if quantity_col and price_col and total_col:

    quantity = pd.to_numeric(
        df[quantity_col],
        errors="coerce"
    )

    price = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    total = pd.to_numeric(
        df[total_col],
        errors="coerce"
    )

    calculated_total = quantity * price

    difference = (
        total - calculated_total
    ).abs()

    consistency_failures = (
        difference > 0.01
    ).sum()


# ------------------------------------------------------------
# 8. DATA QUALITY - FRESHNESS
# ------------------------------------------------------------

latest_date = None
data_age_days = np.nan

if date_col:

    dates = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    latest_date = dates.max()

    if pd.notna(latest_date):

        today = pd.Timestamp.today().normalize()

        data_age_days = (
            today - latest_date.normalize()
        ).days


# ------------------------------------------------------------
# 9. QUALITY CONTRACT
# ------------------------------------------------------------

overall_missing_rate = (
    df.isnull().mean().mean() * 100
)

if overall_missing_rate <= 1:
    completeness_status = "PASS"
else:
    completeness_status = "FAIL"


if pd.notna(duplicate_rate):

    if duplicate_rate <= 0.1:
        uniqueness_status = "PASS"
    else:
        uniqueness_status = "FAIL"

else:

    uniqueness_status = "NOT CHECKED"


if quantity_col or price_col or total_col:

    validity_errors = (
        invalid_quantity +
        invalid_price +
        invalid_total
    )

    validity_status = (
        "PASS"
        if validity_errors == 0
        else "FAIL"
    )

else:

    validity_status = "NOT CHECKED"


if pd.notna(consistency_failures):

    consistency_rate = (
        consistency_failures / len(df) * 100
    )

    consistency_status = (
        "PASS"
        if consistency_rate <= 0.1
        else "FAIL"
    )

else:

    consistency_rate = np.nan
    consistency_status = "NOT CHECKED"


if pd.notna(data_age_days):

    freshness_status = (
        "PASS"
        if data_age_days <= 2
        else "FAIL"
    )

else:

    freshness_status = "NOT CHECKED"


quality_report = pd.DataFrame({

    "Dimension": [
        "Completeness",
        "Uniqueness",
        "Validity",
        "Consistency",
        "Freshness"
    ],

    "Result": [
        f"{overall_missing_rate:.2f}% missing",
        f"{duplicate_rate:.2f}% duplicates"
        if pd.notna(duplicate_rate)
        else "N/A",

        f"{validity_errors} invalid records"
        if quantity_col or price_col or total_col
        else "N/A",

        f"{consistency_rate:.2f}% inconsistent"
        if pd.notna(consistency_rate)
        else "N/A",

        f"{data_age_days} days old"
        if pd.notna(data_age_days)
        else "N/A"
    ],

    "Threshold": [
        "<= 1%",
        "<= 0.1%",
        "0 invalid",
        "<= 0.1%",
        "<= 2 days"
    ],

    "Status": [
        completeness_status,
        uniqueness_status,
        validity_status,
        consistency_status,
        freshness_status
    ]
})


print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

display(quality_report)


# ------------------------------------------------------------
# 10. KPI CALCULATIONS
# ------------------------------------------------------------

total_revenue = np.nan
total_orders = np.nan
average_order_value = np.nan
units_sold = np.nan
unique_customers = np.nan
revenue_per_customer = np.nan
average_unit_price = np.nan
discount_order_rate = np.nan


if total_col:

    total_values = pd.to_numeric(
        df[total_col],
        errors="coerce"
    )

    total_revenue = total_values.sum()


if order_col:

    total_orders = df[order_col].nunique()


if total_orders and total_orders > 0:

    average_order_value = (
        total_revenue / total_orders
    )


if quantity_col:

    quantity_values = pd.to_numeric(
        df[quantity_col],
        errors="coerce"
    )

    units_sold = quantity_values.sum()


if customer_col:

    unique_customers = (
        df[customer_col]
        .nunique()
    )


if unique_customers and unique_customers > 0:

    revenue_per_customer = (
        total_revenue / unique_customers
    )


if price_col:

    price_values = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    average_unit_price = price_values.mean()


if discount_col and order_col:

    discounted = (
        pd.to_numeric(
            df[discount_col],
            errors="coerce"
        ) > 0
    )

    discounted_orders = (
        df.loc[discounted, order_col]
        .nunique()
    )

    discount_order_rate = (
        discounted_orders /
        total_orders * 100
    )


# ------------------------------------------------------------
# 11. KPI DICTIONARY
# ------------------------------------------------------------

kpi_dictionary = pd.DataFrame({

    "KPI": [
        "Total Revenue",
        "Total Orders",
        "Average Order Value",
        "Units Sold",
        "Unique Customers",
        "Revenue per Customer",
        "Discount Order Rate",
        "Average Unit Price",
        "Revenue by Category",
        "Repeat Customer Rate"
    ],

    "Business Definition": [
        "Total monetary value generated from valid retail transactions",
        "Number of unique retail orders/transactions",
        "Average revenue generated per order",
        "Total quantity of products sold",
        "Number of distinct customers who purchased",
        "Average revenue generated per customer",
        "Percentage of orders receiving a discount",
        "Average selling price per unit",
        "Revenue generated by each product category",
        "Percentage of customers who placed more than one order"
    ],

    "Formula": [
        "SUM(total_spent)",
        "COUNT DISTINCT(order_id)",
        "Total Revenue / Total Orders",
        "SUM(quantity)",
        "COUNT DISTINCT(customer_id)",
        "Total Revenue / Unique Customers",
        "Discounted Orders / Total Orders × 100",
        "SUM(price) / COUNT(price)",
        "SUM(total_spent) GROUP BY category",
        "Repeat Customers / Unique Customers × 100"
    ],

    "Grain": [
        "Order",
        "Order",
        "Order",
        "Order Line",
        "Customer",
        "Customer",
        "Order",
        "Order Line",
        "Category",
        "Customer"
    ],

    "Filters": [
        "Valid orders",
        "Valid orders",
        "Valid orders",
        "Valid orders",
        "Valid customers",
        "Valid customers",
        "Valid orders",
        "Valid orders",
        "Valid orders",
        "Valid customers"
    ],

    "Owner": [
        "Retail Operations Manager",
        "Retail Operations Manager",
        "Finance Manager",
        "Inventory Manager",
        "Sales Manager",
        "Sales Manager",
        "Marketing Manager",
        "Pricing Manager",
        "Category Manager",
        "CRM Manager"
    ],

    "Refresh Cadence": [
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Weekly",
        "Daily",
        "Daily",
        "Daily",
        "Weekly"
    ]
})


# ------------------------------------------------------------
# 12. KPI RESULTS
# ------------------------------------------------------------

kpi_results = pd.DataFrame({

    "KPI": [
        "Total Revenue",
        "Total Orders",
        "Average Order Value",
        "Units Sold",
        "Unique Customers",
        "Revenue per Customer",
        "Discount Order Rate",
        "Average Unit Price"
    ],

    "Value": [
        total_revenue,
        total_orders,
        average_order_value,
        units_sold,
        unique_customers,
        revenue_per_customer,
        discount_order_rate,
        average_unit_price
    ]
})


print("\n" + "=" * 70)
print("KPI RESULTS")
print("=" * 70)

display(kpi_results)


# ------------------------------------------------------------
# 13. DATA QUALITY CONTRACT
# ------------------------------------------------------------

contract = pd.DataFrame({

    "Quality Dimension": [
        "Completeness",
        "Uniqueness",
        "Validity",
        "Consistency",
        "Freshness"
    ],

    "Rule": [
        "Required fields must have <= 1% missing values",
        "Duplicate transaction rate must be <= 0.1%",
        "Quantity and price must be positive; total must not be negative",
        "total_spent should match price × quantity within ₹0.01",
        "Latest data must not be more than 2 days old"
    ],

    "Failure Action": [
        "Notify data owner and investigate missing fields",
        "Pause affected KPI and investigate duplicate transactions",
        "Correct invalid records before KPI publication",
        "Investigate financial calculation discrepancies",
        "Escalate to data owner and investigate data pipeline"
    ]
})


# ------------------------------------------------------------
# 14. SAVE EVERYTHING TO EXCEL
# ------------------------------------------------------------

output_file = "KPI_Dictionary_and_Data_Quality.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    kpi_dictionary.to_excel(
        writer,
        sheet_name="KPI Dictionary",
        index=False
    )

    kpi_results.to_excel(
        writer,
        sheet_name="KPI Results",
        index=False
    )

    completeness.to_excel(
        writer,
        sheet_name="Completeness",
        index=False
    )

    quality_report.to_excel(
        writer,
        sheet_name="Quality Report",
        index=False
    )

    contract.to_excel(
        writer,
        sheet_name="Quality Contract",
        index=False
    )


# ------------------------------------------------------------
# 15. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROJECT COMPLETED")
print("=" * 70)

print("\nCreated file:")
print(output_file)

print("\nProject contains:")
print("✓ KPI Dictionary")
print("✓ KPI Results")
print("✓ Completeness Check")
print("✓ Uniqueness Check")
print("✓ Validity Check")
print("✓ Consistency Check")
print("✓ Freshness Check")
print("✓ Data Quality Contract")

print("\nFinal Quality Status:")

if all(
    status in ["PASS", "NOT CHECKED"]
    for status in quality_report["Status"]
):
    print("PASS - Data meets the defined quality contract.")
else:
    print("FAIL - Data quality issues require investigation.")

print("\nDownload the Excel file from the Colab Files panel.")

RETAIL KPI & DATA QUALITY PROJECT

Rows    : 12
Columns : 9

Column Names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Standardized Columns:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Detected Columns:
Order     : order_id
Customer  : None
Quantity  : quantity
Price     : unit_price
Total     : None
Date      : order_date
Category  : category
Discount  : None

COMPLETENESS CHECK


,Column,Missing Count,Missing %
0,order_id,0,0.000000
1,order_date,1,8.333333
2,customer_segment,0,0.000000
3,city,1,8.333333
4,category,0,0.000000
5,quantity,0,0.000000
6,unit_price,0,0.000000
7,discount_pct,1,8.333333
8,payment_status,0,0.000000



DATA QUALITY REPORT


,Dimension,Result,Threshold,Status
0,Completeness,2.78% missing,<= 1%,FAIL
1,Uniqueness,8.33% duplicates,<= 0.1%,FAIL
2,Validity,1 invalid records,0 invalid,FAIL
3,Consistency,N/A,<= 0.1%,NOT CHECKED
4,Freshness,241 days old,<= 2 days,FAIL



KPI RESULTS


,KPI,Value
0,Total Revenue,NaN
1,Total Orders,11.000000
2,Average Order Value,NaN
3,Units Sold,16.000000
4,Unique Customers,NaN
5,Revenue per Customer,NaN
6,Discount Order Rate,NaN
7,Average Unit Price,1082.333333



PROJECT COMPLETED

Created file:
KPI_Dictionary_and_Data_Quality.xlsx

Project contains:
✓ KPI Dictionary
✓ KPI Results
✓ Completeness Check
✓ Uniqueness Check
✓ Validity Check
✓ Consistency Check
✓ Freshness Check
✓ Data Quality Contract

Final Quality Status:
FAIL - Data quality issues require investigation.

Download the Excel file from the Colab Files panel.
